# Personal chef: combining prompts, search and memory

## Goal

Build a recipe assistant with instructions, built-in web search and conversation memory.

## Setup

Select the **Python (LangChain OpenAI)** kernel and run cells from top to bottom.
`OPENAI_API_KEY` and `OPENAI_MODEL` are read only from this course's root `.env`.
Shared settings are in `lesson_support.py`: up to 1024 output tokens, a 30-second
HTTP timeout, no HTTP retries, and LangSmith tracing disabled.
`make_model("another-api-model-id")` creates a different model with the same key.
The default text settings are intended for `gpt-5.6-luna`.
Creating a model does not call the API; `invoke` and consuming `stream` make paid requests.

Web search is billed separately. InMemorySaver retains this chef conversation only until the kernel stops.


In [ ]:
from pathlib import Path
import sys

# Find this course's root, whether Jupyter starts at the root or in notebooks/.
current_dir = Path.cwd().resolve()
course_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "course.json").is_file() and (folder / "lesson_support.py").is_file()),
    None,
)
if course_root is None:
    raise RuntimeError("Start Jupyter from the course root, as described in README.md.")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from lesson_support import make_model

model = make_model()
print("Model:", model.model_name)


In [ ]:
# Built-in OpenAI search: no separate Tavily API key is needed.
search_model = make_model(use_responses_api=True)
web_search_tool = {"type": "web_search"}


In [ ]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=search_model,
    tools=[web_search_tool],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].text)

In [ ]:
from pprint import pprint

pprint(response)